In [1]:
!pip install -q qdrant-client sentence-transformers requests groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.5 MB/s eta 0:00:00


In [3]:
import os
import requests
github_url =  "https://github.com/LavanyaKommera/ai-engineer-portfolio/blob/main/Vitaminsandmineralstxt.txt"

def load_document(url: str) -> str:
    """Fetch a plain-text file from a raw GitHub URL."""
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.text

raw_text = load_document(github_url)
print(f"Loaded {len(raw_text):,} characters")
print(raw_text[:400])  # Sanity check

Loaded 276,408 characters






<!DOCTYPE html>
<html
  lang="en"
  
  data-color-mode="auto" data-light-theme="light" data-dark-theme="dark"
  data-a11y-animated-images="system" data-a11y-link-underlines="true"
  
  >




  <head>
    <meta charset="utf-8">
  <link rel="dns-prefetch" href="https://github.githubassets.com">
  <link rel="dns-prefetch" href="https://avatars.githubusercontent.com">
  <link rel="dns-prefetch" 


In [9]:
CHUNK_SIZE = 100

def parse_word_chunks(text: str, chunk_size: int = CHUNK_SIZE) -> list[dict]:
    # Strip markdown heading symbols and blank lines
    clean_lines = []
    for line in text.splitlines():
        line = line.strip().lstrip("#").strip()
        if line:
            clean_lines.append(line)

    # Join everything into one word list and slice
    words = " ".join(clean_lines).split()

    chunks = []
    for i in range(0, len(words), chunk_size):
        content = " ".join(words[i : i + chunk_size])
        chunks.append({
            "chunk_index": len(chunks),
            "content": content,
        })

    return chunks

In [10]:
chunks = parse_word_chunks(raw_text)
print(f"Total chunks: {len(chunks)}")

Total chunks: 176


In [11]:
for chunk in chunks[:5]:
    print("─" * 55)
    print(f"Content : {chunk['content'][:200]}…")

───────────────────────────────────────────────────────
Content : <!DOCTYPE html> <html lang="en" data-color-mode="auto" data-light-theme="light" data-dark-theme="dark" data-a11y-animated-images="system" data-a11y-link-underlines="true" > <head> <meta charset="utf-8…
───────────────────────────────────────────────────────
Content : /><link data-color-theme="dark_colorblind" crossorigin="anonymous" media="all" rel="stylesheet" data-href="https://github.githubassets.com/assets/dark_colorblind-37023bf69d8e0e34.css" /><link data-col…
───────────────────────────────────────────────────────
Content : crossorigin="anonymous" type="application/javascript" src="https://github.githubassets.com/assets/fetch-utilities-f1b06a7448671df6.js" defer="defer"></script> <script crossorigin="anonymous" type="app…
───────────────────────────────────────────────────────
Content : crossorigin="anonymous" type="application/javascript" src="https://github.githubassets.com/assets/79039-f2b81734929d0b15.js" defer

In [12]:
def build_chunk_text(chunk: dict) -> str:
    return chunk["content"]

In [13]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
# Extract Chunk Texts
chunk_texts = [build_chunk_text(c) for c in chunks]

print(f"Embedding {len(chunk_texts)} chunks …")
embeddings = embedder.encode(chunk_texts, show_progress_bar=True)

print(f"Shape: {embeddings.shape}")

Embedding 176 chunks …


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Shape: (176, 384)


In [16]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 903.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentele

In [18]:
import chromadb
client = chromadb.PersistentClient("./tmp/chromadb")

COLLECTION_NAME = "vitamindocs"
DIM = embedder.get_embedding_dimension()

client.delete_collection("vitaminkb_collection")

collection = client.get_or_create_collection(
    name="vitaminkb_collection",
)
print("Collection created.", collection.name)

Collection created. vitaminkb_collection


In [19]:
# Creating Points

#start ID sequence
ids = [f"chunk_{i}" for i in range(len(chunk_texts))]

metadatas = [
    {
        "chunk_index": i,
        "source": "insurance_rag_knowledge_base.txt"
    }
    for i in range(len(chunk_texts))
]

if collection.count() == 0:
    collection.add(
        ids=ids,
        documents=chunk_texts,
        embeddings=embeddings.tolist(),
        metadatas=metadatas
    )
    print(f"Inserted {len(chunk_texts)} chunks")
else:
    print(f"Collection already has {collection.count()} records — skipping insert")


Inserted 176 chunks


In [21]:
print(f"collection count: {collection.count()}")

sample = collection.get(limit=1, include=["embeddings"])
print(f"Dimensions : {len(sample['embeddings'][0])}")

collection count: 176
Dimensions : 384


In [24]:
def retrieve(
    query: str,
    top_k: int = 5
) -> list[dict]:
    """
    Embed the query and return the top-k most similar chunks.

    Args:
        query          : User's question.
        top_k          : Number of chunks to return.
        section_filter : Optional H2 heading to restrict the search scope.
    """
    query_vector = embedder.encode(query).tolist()

    hits = collection.query(
        query_embeddings=query_vector,
        n_results=top_k,
        include=["documents","metadatas","distances"]
    )
    retrieved_chunks = []
    if hits and hits['documents'] and len(hits['documents']) > 0:
        for i in range(len(hits['documents'][0])):
            chunk = {
                "content": hits['documents'][0][i],
                "score": round(hits['distances'][0][i], 4),
                **hits['metadatas'][0][i]
            }
            retrieved_chunks.append(chunk)

    return retrieved_chunks

In [25]:
results = retrieve("What is the source of Riboflavin? ")
for r in results:
    print(f"[score={r['score']}]")
    print(f"  {r['content'][:200]}…\n")

[score=1.1076]
  Whole grains, eggs, soybeans, fish","","B-9: Fortified grains and cereals, asparagus, spinach, broccoli, legumes (black-eyed peas and chickpeas), orange juice","","B-12: Meat, poultry, fish, milk, che…

[score=1.1222]
  is essential for the metabolism of food. It also plays a role in the production of hormones and cholesterol."," Riboflavin (vitamin B2) works with the other B vitamins. It is important for body growth…

[score=1.227]
  ","Fat-soluble vitamins"," These are stored in the body's cells. They are not passed out of the body as easily as water-soluble vitamins. Fat-soluble vitamins can reach toxic levels if you get more th…

[score=1.286]
  health (osteoporosis).",""," Vitamin A helps form and maintain healthy teeth, bones, soft tissue, mucous membranes, skin, and tissue in the retina (the back of the eye, which creates vision)."," Vitam…

[score=1.2874]
  of proteins and carbohydrates, and in the production of hormones and cholesterol."," Niacin is a B vitami

In [26]:
SYSTEM_PROMPT = """You are a helpful HR assistant.
Answer the user's question using ONLY the context provided below.
If the context does not contain enough information, say so — do not make things up.
Always cite the section name when referencing specific information."""

In [27]:
def build_context(retrieved_chunks: list[dict]) -> str:
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        parts.append(f"[Source {i}]\n{chunk['content']}")
    return "\n\n---\n\n".join(parts)

In [29]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [30]:
from groq import Groq

groq_client = Groq()   # Reads GROQ_API_KEY from environment automatically
GROQ_MODEL  = "openai/gpt-oss-safeguard-20b"

def rag(query: str, top_k: int = 5):
    """
    End-to-end RAG pipeline:
      1. Retrieve relevant chunks from Qdrant
      2. Format them as a context block
      3. Send context + query to Groq and return the answer
    """
    # Step 1 — Retrieve
    chunks = retrieve(query, top_k=top_k)
    if not chunks:
        return "No relevant content found in the document."

    # Step 2 — Build context
    context = build_context(chunks)

    # Step 3 — Generate
    user_message = f"Context:\n{context}\n\nQuestion: {query}"

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.2,   # Low = factual;  High = creative
    )
    return response.choices[0].message.content, context

In [31]:
answer, context = rag("Why is Riboflavin important")
print(answer)
print(f"{250*'='}")
print(f"\n\nSOURCES:\n {context}")

Riboflavin (vitamin B2) is important because it works with the other B vitamins to support the body’s growth and to help produce red blood cells【Source 1】.


SOURCES:
 [Source 1]
is essential for the metabolism of food. It also plays a role in the production of hormones and cholesterol."," Riboflavin (vitamin B2) works with the other B vitamins. It is important for body growth and the production of red blood cells."," Thiamine (vitamin B1) helps the body cells change carbohydrates into energy. Getting enough carbohydrates is very important during pregnancy and breastfeeding. It is also essential for heart function and healthy nerve cells."," Choline helps in normal functioning of the brain and nervous system. Lack of choline can cause swelling in the liver."," Carnitine helps the body to change fatty

---

[Source 2]
health (osteoporosis).",""," Vitamin A helps form and maintain healthy teeth, bones, soft tissue, mucous membranes, skin, and tissue in the retina (the back of the eye, wh